# Notebook 122: BPTT ― 時間を遡る逆伝播
## Backpropagation Through Time: Vanishing & Exploding Gradients
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の理論的中核。RNNの逆伝播アルゴリズムBPTTを完全にNumPyで実装し、勾配消失・爆発問題を実証します。

### 学習目標
1. **BPTTの数式** を導出し、NumPyで完全に実装できる
2. **勾配消失・爆発** のメカニズムを固有値の観点から理解する
3. **Truncated BPTT** を実装し、メモリ効率の改善を体験する
4. **勾配クリッピング** を実装できる
5. **Adding Problem** でRNNの長期依存の限界を実証できる

### 前提知識
- Notebook 70-76（ニューラルエンジンの深部、特にNB73-74の逆伝播）
- Notebook 120-121（シーケンス導入、バニラRNN）

⏱️ **推定学習時間**: 150-180分  
📊 **難易度**: ★★★★☆（上級）  
🎓 **カテゴリ**: 理論・実装

## 目次

1. [BPTTの数式導出](#1-bpttの数式導出)
2. [BPTTの完全実装](#2-bpttの完全実装)
3. [勾配消失・爆発のメカニズム](#3-勾配消失爆発のメカニズム)
4. [勾配の流れの可視化](#4-勾配の流れの可視化)
5. [Truncated BPTT](#5-truncated-bptt)
6. [勾配クリッピング](#6-勾配クリッピング)
7. [Adding Problem（Hochreiter 1997）](#7-adding-problemhochreiter-1997)
8. [まとめと次のステップ](#8-まとめと次のステップ)

In [ ]:
# ============================================================
# 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（環境に応じて調整）
plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# 再現性のためのシード
np.random.seed(42)

print("環境セットアップ完了")
print(f"NumPy version: {np.__version__}")
print()
print("このノートブックでは NumPy のみを使用して")
print("BPTT（Backpropagation Through Time）を完全に実装します。")

---
## 1. BPTTの数式導出

### 1.1 RNNの順伝播の復習

RNN（Recurrent Neural Network）は、時刻 $t$ における隠れ状態 $h_t$ を次のように更新します：

$$
h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

出力は：

$$
y_t = W_{hy} h_t + b_y
$$

ここで：
- $x_t \in \mathbb{R}^{d}$ : 時刻 $t$ の入力
- $h_t \in \mathbb{R}^{H}$ : 時刻 $t$ の隠れ状態
- $W_{xh} \in \mathbb{R}^{d \times H}$ : 入力→隠れ層の重み
- $W_{hh} \in \mathbb{R}^{H \times H}$ : 隠れ層→隠れ層の重み（再帰重み）
- $W_{hy} \in \mathbb{R}^{H \times o}$ : 隠れ層→出力の重み

### 1.2 損失関数

系列全体の損失は、各時刻の損失の和です：

$$
L = \sum_{t=1}^{T} L_t
$$

### 1.3 BPTTの核心：時間を通じた連鎖律

$W_{hh}$ に関する勾配を求めるとき、$h_t$ が $W_{hh}$ に **間接的に** 依存していることが重要です：

$$
h_t \rightarrow h_{t-1} \rightarrow h_{t-2} \rightarrow \cdots \rightarrow h_1
$$

したがって、連鎖律を適用すると：

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=1}^{t} \frac{\partial L_t}{\partial h_t} \left( \prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} \right) \frac{\partial h_k}{\partial W_{hh}}
$$

### 1.4 ヤコビアン行列

隠れ状態間のヤコビアンは：

$$
\frac{\partial h_j}{\partial h_{j-1}} = \text{diag}(1 - h_j^2) \cdot W_{hh}
$$

ここで $\text{diag}(1 - h_j^2)$ は $\tanh$ の導関数を対角行列にしたものです。

### 1.5 勾配積の問題

ヤコビアンの積 $\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}}$ が BPTT の本質的な問題を引き起こします：

- $t - k$ が大きい（長い時間を遡る）場合、この積は **指数関数的に** 減衰（消失）または増大（爆発）する
- これが **勾配消失問題** と **勾配爆発問題** です

In [ ]:
# ============================================================
# Section 2: BPTTの完全実装
# ============================================================

class RNNCell:
    """単一時刻のRNNセル（順伝播と逆伝播）"""
    
    def __init__(self, input_dim, hidden_dim):
        scale = np.sqrt(2.0 / (input_dim + hidden_dim))
        self.Wxh = np.random.randn(input_dim, hidden_dim) * scale
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.bh = np.zeros(hidden_dim)
    
    def forward(self, x_t, h_prev):
        """順伝播：1時刻分
        x_t: (batch, input_dim)
        h_prev: (batch, hidden_dim)
        """
        self.cache = (x_t, h_prev)
        z = x_t @ self.Wxh + h_prev @ self.Whh + self.bh
        h_next = np.tanh(z)
        self.cache_h = h_next
        return h_next
    
    def backward(self, dh_next):
        """逆伝播：1時刻分
        dh_next: (batch, hidden_dim) 上流からの勾配
        """
        x_t, h_prev = self.cache
        h_next = self.cache_h
        
        # tanh の導関数: dtanh/dz = 1 - tanh^2(z)
        dtanh = dh_next * (1 - h_next ** 2)
        
        # 各パラメータの勾配
        self.dWxh = x_t.T @ dtanh
        self.dWhh = h_prev.T @ dtanh
        self.dbh = np.sum(dtanh, axis=0)
        
        # 入力と前の隠れ状態への勾配
        dx = dtanh @ self.Wxh.T
        dh_prev = dtanh @ self.Whh.T
        return dx, dh_prev


class RNNWithBPTT:
    """BPTT付きのフルRNN実装"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Xavier初期化
        scale_xh = np.sqrt(2.0 / (input_dim + hidden_dim))
        scale_hh = np.sqrt(2.0 / (hidden_dim + hidden_dim))
        scale_hy = np.sqrt(2.0 / (hidden_dim + output_dim))
        
        self.Wxh = np.random.randn(input_dim, hidden_dim) * scale_xh
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * scale_hh
        self.bh = np.zeros(hidden_dim)
        self.Why = np.random.randn(hidden_dim, output_dim) * scale_hy
        self.by = np.zeros(output_dim)
    
    def forward(self, X_seq):
        """順伝播
        X_seq: (T, batch, input_dim)
        returns: outputs (T, batch, output_dim), hidden_states list
        """
        T, batch, _ = X_seq.shape
        
        self.T = T
        self.batch = batch
        self.X_seq = X_seq
        
        # 隠れ状態の保存
        self.h_states = [np.zeros((batch, self.hidden_dim))]  # h_0
        self.z_cache = []  # tanh前の値を保存
        
        outputs = []
        
        for t in range(T):
            x_t = X_seq[t]  # (batch, input_dim)
            h_prev = self.h_states[-1]
            
            z = x_t @ self.Wxh + h_prev @ self.Whh + self.bh
            h_next = np.tanh(z)
            
            self.z_cache.append(z)
            self.h_states.append(h_next)
            
            # 出力
            y_t = h_next @ self.Why + self.by
            outputs.append(y_t)
        
        return np.array(outputs)  # (T, batch, output_dim)
    
    def backward(self, dL_doutput):
        """BPTT: 全時刻を通じた逆伝播
        dL_doutput: (T, batch, output_dim) 各時刻の出力に対する損失の勾配
        """
        T = self.T
        
        # 勾配の初期化
        self.dWxh = np.zeros_like(self.Wxh)
        self.dWhh = np.zeros_like(self.Whh)
        self.dbh = np.zeros_like(self.bh)
        self.dWhy = np.zeros_like(self.Why)
        self.dby = np.zeros_like(self.by)
        
        # 次の時刻からの勾配（最後の時刻は0で初期化）
        dh_next = np.zeros((self.batch, self.hidden_dim))
        
        # 勾配ノルムの記録（可視化用）
        self.grad_norms = []
        
        # 時間を逆方向に遡る
        for t in reversed(range(T)):
            # 出力層の勾配
            dy_t = dL_doutput[t]  # (batch, output_dim)
            self.dWhy += self.h_states[t + 1].T @ dy_t
            self.dby += np.sum(dy_t, axis=0)
            
            # 隠れ状態への勾配 = 出力層からの勾配 + 次の時刻からの勾配
            dh = dy_t @ self.Why.T + dh_next
            
            # 勾配ノルムを記録
            self.grad_norms.append(np.linalg.norm(dh))
            
            # tanh の逆伝播
            h_t = self.h_states[t + 1]
            dtanh = dh * (1 - h_t ** 2)
            
            # パラメータ勾配の蓄積
            x_t = self.X_seq[t]
            h_prev = self.h_states[t]
            
            self.dWxh += x_t.T @ dtanh
            self.dWhh += h_prev.T @ dtanh
            self.dbh += np.sum(dtanh, axis=0)
            
            # 前の時刻への勾配伝播
            dh_next = dtanh @ self.Whh.T
        
        # 勾配ノルムを時間順に並べ替え
        self.grad_norms = self.grad_norms[::-1]
    
    def clip_gradients(self, max_norm):
        """勾配クリッピング（ノルムベース）"""
        grads = [self.dWxh, self.dWhh, self.dbh, self.dWhy, self.dby]
        total_norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
        
        clip_coeff = max_norm / (total_norm + 1e-6)
        if clip_coeff < 1:
            self.dWxh *= clip_coeff
            self.dWhh *= clip_coeff
            self.dbh *= clip_coeff
            self.dWhy *= clip_coeff
            self.dby *= clip_coeff
        
        return total_norm
    
    def update(self, lr):
        """SGDによるパラメータ更新"""
        self.Wxh -= lr * self.dWxh
        self.Whh -= lr * self.dWhh
        self.bh -= lr * self.dbh
        self.Why -= lr * self.dWhy
        self.by -= lr * self.dby


# テスト: 簡単な順伝播と逆伝播
np.random.seed(42)
rnn = RNNWithBPTT(input_dim=3, hidden_dim=8, output_dim=1)

# ダミーデータ: T=5, batch=4, input_dim=3
T, batch = 5, 4
X = np.random.randn(T, batch, 3)
targets = np.random.randn(T, batch, 1)

# 順伝播
outputs = rnn.forward(X)

# MSE損失
loss = np.mean((outputs - targets) ** 2)
dL_doutput = 2.0 * (outputs - targets) / (T * batch)

# 逆伝播
rnn.backward(dL_doutput)

print("RNNWithBPTT の基本テスト")
print("=" * 50)
print(f"入力形状:   X = ({T}, {batch}, 3)")
print(f"出力形状:   outputs = {outputs.shape}")
print(f"損失:       L = {loss:.6f}")
print(f"")
print(f"勾配の形状:")
print(f"  dWxh: {rnn.dWxh.shape}  norm = {np.linalg.norm(rnn.dWxh):.6f}")
print(f"  dWhh: {rnn.dWhh.shape}  norm = {np.linalg.norm(rnn.dWhh):.6f}")
print(f"  dbh:  {rnn.dbh.shape}   norm = {np.linalg.norm(rnn.dbh):.6f}")
print(f"  dWhy: {rnn.dWhy.shape}  norm = {np.linalg.norm(rnn.dWhy):.6f}")
print(f"  dby:  {rnn.dby.shape}   norm = {np.linalg.norm(rnn.dby):.6f}")

In [ ]:
# ============================================================
# 数値勾配チェック: BPTTの実装を検証
# ============================================================

def numerical_gradient_check(rnn, X, targets, param_name, h=1e-5):
    """数値微分によるBPTTの勾配検証"""
    param = getattr(rnn, param_name)
    numerical_grad = np.zeros_like(param)
    
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        original = param[idx]
        
        # f(x + h)
        param[idx] = original + h
        out_plus = rnn.forward(X)
        loss_plus = np.mean((out_plus - targets) ** 2)
        
        # f(x - h)
        param[idx] = original - h
        out_minus = rnn.forward(X)
        loss_minus = np.mean((out_minus - targets) ** 2)
        
        # 中心差分
        numerical_grad[idx] = (loss_plus - loss_minus) / (2 * h)
        
        # 復元
        param[idx] = original
        it.iternext()
    
    return numerical_grad


# 小さなRNNで勾配チェック
np.random.seed(42)
rnn_check = RNNWithBPTT(input_dim=2, hidden_dim=3, output_dim=1)

T_check, batch_check = 4, 2
X_check = np.random.randn(T_check, batch_check, 2) * 0.5
targets_check = np.random.randn(T_check, batch_check, 1) * 0.5

# 解析的勾配を計算
outputs_check = rnn_check.forward(X_check)
dL = 2.0 * (outputs_check - targets_check) / (T_check * batch_check)
rnn_check.backward(dL)

print("数値勾配チェック（BPTT実装の検証）")
print("=" * 60)

for param_name in ['Wxh', 'Whh', 'bh', 'Why', 'by']:
    # 数値勾配
    num_grad = numerical_gradient_check(rnn_check, X_check, targets_check, param_name)
    # 解析的勾配
    ana_grad = getattr(rnn_check, 'd' + param_name)
    
    # 再計算（数値勾配チェックでforwardが呼ばれたため）
    outputs_check = rnn_check.forward(X_check)
    dL = 2.0 * (outputs_check - targets_check) / (T_check * batch_check)
    rnn_check.backward(dL)
    ana_grad = getattr(rnn_check, 'd' + param_name)
    
    # 相対誤差
    diff = np.abs(num_grad - ana_grad)
    denom = np.maximum(np.abs(num_grad) + np.abs(ana_grad), 1e-8)
    relative_error = np.max(diff / denom)
    
    status = "OK" if relative_error < 1e-5 else "NG"
    print(f"  d{param_name:3s}: 最大相対誤差 = {relative_error:.2e}  [{status}]")

print("\n勾配チェック完了。誤差が 1e-5 以下であればBPTT実装は正しいです。")

---
## 3. 勾配消失・爆発のメカニズム

### 3.1 ヤコビアン行列の分析

隠れ状態 $h_t$ から $h_{t-1}$ への逆伝播では、次のヤコビアンが掛かります：

$$
\frac{\partial h_t}{\partial h_{t-1}} = \text{diag}(1 - h_t^2) \cdot W_{hh}
$$

$k$ 時刻分遡るとき、ヤコビアンの積になります：

$$
\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} = \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}
$$

### 3.2 固有値との関係

$W_{hh}$ の **スペクトル半径**（最大固有値の絶対値）を $\rho(W_{hh})$ とすると：

- $\rho(W_{hh}) < 1$ の場合: ヤコビアン積 → **指数的に減衰（勾配消失）**
- $\rho(W_{hh}) > 1$ の場合: ヤコビアン積 → **指数的に増大（勾配爆発）**

ただし $\tanh$ の導関数（$1 - h_t^2 \in (0, 1]$）が常に $\leq 1$ であることも重要です。これは勾配を **さらに** 小さくする方向に働きます。

### 3.3 直感的理解

| 状態 | $\rho(W_{hh})$ | 勾配の挙動 | 結果 |
|------|----------------|-----------|------|
| 消失 | $< 1$ | 指数的減衰 | 長期記憶を学習できない |
| 境界 | $\approx 1$ | 安定 | 理想的（達成困難） |
| 爆発 | $> 1$ | 指数的増大 | 学習が不安定/発散 |

In [ ]:
# ============================================================
# 固有値分析と勾配の流れの可視化
# ============================================================

np.random.seed(42)

def analyze_gradient_flow(Whh, T=50, n_trials=20):
    """Whhの固有値に基づく勾配の流れを分析する
    
    ヤコビアン積 prod_{j=k+1}^{t} diag(1-h_j^2) * Whh をシミュレート
    """
    hidden_dim = Whh.shape[0]
    grad_norms = np.zeros(T)
    
    for trial in range(n_trials):
        # ランダムな隠れ状態を生成（tanh出力なので[-1,1]）
        h_states = [np.tanh(np.random.randn(hidden_dim) * 0.5) for _ in range(T)]
        
        # 初期勾配（最後の時刻）
        grad = np.random.randn(hidden_dim)
        grad = grad / np.linalg.norm(grad)  # 正規化
        
        norms = [np.linalg.norm(grad)]
        
        # 時間を遡ってヤコビアンを適用
        for t in reversed(range(T - 1)):
            h_t = h_states[t + 1]
            # ヤコビアン: diag(1 - h_t^2) * Whh
            dtanh = 1 - h_t ** 2
            grad = (grad * dtanh) @ Whh.T
            norms.append(np.linalg.norm(grad))
        
        grad_norms += np.array(norms[::-1])
    
    return grad_norms / n_trials


# 異なるスペクトル半径のWhhを生成
hidden_dim = 32

def make_Whh_with_spectral_radius(dim, target_radius):
    """指定したスペクトル半径のWhh行列を生成"""
    W = np.random.randn(dim, dim)
    eigenvalues = np.linalg.eigvals(W)
    current_radius = np.max(np.abs(eigenvalues))
    W = W * (target_radius / current_radius)
    return W


spectral_radii = [0.5, 0.9, 1.0, 1.2, 1.5]
colors = ['blue', 'green', 'orange', 'red', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左: 勾配ノルムの推移
for sr, color in zip(spectral_radii, colors):
    np.random.seed(42)
    Whh = make_Whh_with_spectral_radius(hidden_dim, sr)
    actual_sr = np.max(np.abs(np.linalg.eigvals(Whh)))
    grad_norms = analyze_gradient_flow(Whh, T=50)
    
    axes[0].semilogy(range(len(grad_norms)), grad_norms, 
                     color=color, linewidth=2,
                     label=f'$\\rho(W_{{hh}})$ = {actual_sr:.1f}')

axes[0].set_xlabel('時刻（過去方向 ← 現在）', fontsize=12)
axes[0].set_ylabel('勾配ノルム（対数スケール）', fontsize=12)
axes[0].set_title('時間を遡るにつれて勾配がどう変化するか', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='基準線 (norm=1)')

# 右: 固有値分布のプロット
for sr, color in zip([0.5, 1.0, 1.5], ['blue', 'orange', 'red']):
    np.random.seed(42)
    Whh = make_Whh_with_spectral_radius(hidden_dim, sr)
    eigenvalues = np.linalg.eigvals(Whh)
    
    axes[1].scatter(eigenvalues.real, eigenvalues.imag, 
                    alpha=0.6, s=40, color=color,
                    label=f'$\\rho$ = {sr}')

# 単位円を描画
theta = np.linspace(0, 2 * np.pi, 100)
axes[1].plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=1.5, label='単位円')

axes[1].set_xlabel('実部', fontsize=12)
axes[1].set_ylabel('虚部', fontsize=12)
axes[1].set_title('$W_{hh}$ の固有値分布', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print("【重要な観察】")
print("・スペクトル半径 < 1: 勾配は指数的に減衰（勾配消失）")
print("・スペクトル半径 > 1: 勾配は指数的に増大（勾配爆発）")
print("・単位円の外に固有値があると、勾配が爆発する")

In [ ]:
# ============================================================
# Section 4: 実際の学習中の勾配ノルム観察
# ============================================================

np.random.seed(42)

def generate_sequence_data(n_samples, seq_length, input_dim=1):
    """簡単なシーケンスデータを生成（sin波の予測）"""
    t = np.linspace(0, 4 * np.pi, seq_length + 1)
    data = np.sin(t)
    
    X = np.zeros((seq_length, n_samples, input_dim))
    Y = np.zeros((seq_length, n_samples, 1))
    
    for i in range(n_samples):
        phase = np.random.uniform(0, 2 * np.pi)
        freq = np.random.uniform(0.8, 1.2)
        signal = np.sin(freq * t + phase)
        X[:, i, 0] = signal[:-1]
        Y[:, i, 0] = signal[1:]
    
    return X, Y


# 長い系列でのBPTT
seq_lengths = [10, 30, 50, 100]
hidden_dim = 16
n_samples = 8

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, seq_len in enumerate(seq_lengths):
    ax = axes[idx // 2, idx % 2]
    
    np.random.seed(42)
    rnn = RNNWithBPTT(input_dim=1, hidden_dim=hidden_dim, output_dim=1)
    X, Y = generate_sequence_data(n_samples, seq_len)
    
    # 順伝播
    outputs = rnn.forward(X)
    loss = np.mean((outputs - Y) ** 2)
    dL = 2.0 * (outputs - Y) / (seq_len * n_samples)
    
    # BPTT
    rnn.backward(dL)
    
    # 勾配ノルムをプロット
    times = list(range(len(rnn.grad_norms)))
    ax.semilogy(times, rnn.grad_norms, 'b-o', markersize=3, linewidth=1.5)
    ax.set_xlabel('時刻 t', fontsize=10)
    ax.set_ylabel('勾配ノルム（対数）', fontsize=10)
    ax.set_title(f'T = {seq_len}（系列長）', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # 勾配消失の度合いを数値で表示
    ratio = rnn.grad_norms[0] / (rnn.grad_norms[-1] + 1e-15)
    ax.text(0.05, 0.95, f'先頭/末尾 比率: {ratio:.2e}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('BPTT中の勾配ノルムの推移（各時刻での∂L/∂h_t）',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【観察結果】")
print("・系列が長くなるほど、初期時刻の勾配ノルムが小さくなる")
print("・これが勾配消失: 過去の情報に基づく学習が困難になる")
print("・T=100 では最初の時刻の勾配はほぼゼロに近い")

---
## 5. Truncated BPTT

### 5.1 アイデア

フルBPTTは $T$ 時刻すべてを遡りますが、**Truncated BPTT** は $K$ 時刻分だけ逆伝播します：

- 系列を長さ $K$ のチャンクに分割
- 順伝播は全時刻にわたって行う（隠れ状態は引き継ぐ）
- 逆伝播は各チャンク内だけに限定

### 5.2 トレードオフ

| 項目 | フルBPTT | Truncated BPTT (K) |
|------|---------|-------------------|
| メモリ | $O(T \cdot H)$ | $O(K \cdot H)$ |
| 長期依存 | 理論上可能 | $K$ 時刻まで |
| 計算量 | $O(T^2 \cdot H^2)$ | $O(T \cdot K \cdot H^2)$ |
| 勾配消失 | 深刻 | 軽減（ただし長期は学べない） |

In [ ]:
# ============================================================
# Truncated BPTT の実装と比較
# ============================================================

class RNNWithTruncatedBPTT:
    """Truncated BPTT 付き RNN"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        scale_xh = np.sqrt(2.0 / (input_dim + hidden_dim))
        scale_hh = np.sqrt(2.0 / (hidden_dim + hidden_dim))
        scale_hy = np.sqrt(2.0 / (hidden_dim + output_dim))
        
        self.Wxh = np.random.randn(input_dim, hidden_dim) * scale_xh
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * scale_hh
        self.bh = np.zeros(hidden_dim)
        self.Why = np.random.randn(hidden_dim, output_dim) * scale_hy
        self.by = np.zeros(output_dim)
    
    def train_step(self, X_seq, targets, K, lr, max_grad_norm=5.0):
        """Truncated BPTT によるトレーニングステップ
        X_seq: (T, batch, input_dim)
        targets: (T, batch, output_dim)
        K: truncation length
        """
        T, batch, _ = X_seq.shape
        
        # 隠れ状態の初期化
        h = np.zeros((batch, self.hidden_dim))
        total_loss = 0.0
        
        # チャンクごとに処理
        for chunk_start in range(0, T, K):
            chunk_end = min(chunk_start + K, T)
            chunk_len = chunk_end - chunk_start
            
            # --- 順伝播 ---
            h_states = [h.copy()]  # detach: 前のチャンクの勾配は遡らない
            outputs = []
            
            for t in range(chunk_start, chunk_end):
                x_t = X_seq[t]
                h_prev = h_states[-1]
                z = x_t @ self.Wxh + h_prev @ self.Whh + self.bh
                h_next = np.tanh(z)
                h_states.append(h_next)
                y_t = h_next @ self.Why + self.by
                outputs.append(y_t)
            
            outputs = np.array(outputs)  # (chunk_len, batch, output_dim)
            chunk_targets = targets[chunk_start:chunk_end]
            
            # 損失
            chunk_loss = np.mean((outputs - chunk_targets) ** 2)
            total_loss += chunk_loss * chunk_len / T
            
            # --- チャンク内の逆伝播 ---
            dWxh = np.zeros_like(self.Wxh)
            dWhh = np.zeros_like(self.Whh)
            dbh = np.zeros_like(self.bh)
            dWhy = np.zeros_like(self.Why)
            dby = np.zeros_like(self.by)
            
            dh_next = np.zeros((batch, self.hidden_dim))
            
            for t_local in reversed(range(chunk_len)):
                t_global = chunk_start + t_local
                
                # 出力層の勾配
                dy_t = 2.0 * (outputs[t_local] - chunk_targets[t_local]) / (T * batch)
                dWhy += h_states[t_local + 1].T @ dy_t
                dby += np.sum(dy_t, axis=0)
                
                dh = dy_t @ self.Why.T + dh_next
                
                h_t = h_states[t_local + 1]
                dtanh = dh * (1 - h_t ** 2)
                
                x_t = X_seq[t_global]
                h_prev = h_states[t_local]
                
                dWxh += x_t.T @ dtanh
                dWhh += h_prev.T @ dtanh
                dbh += np.sum(dtanh, axis=0)
                
                dh_next = dtanh @ self.Whh.T
            
            # 勾配クリッピング
            grads = [dWxh, dWhh, dbh, dWhy, dby]
            total_norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
            clip_coeff = max_grad_norm / (total_norm + 1e-6)
            if clip_coeff < 1:
                dWxh *= clip_coeff
                dWhh *= clip_coeff
                dbh *= clip_coeff
                dWhy *= clip_coeff
                dby *= clip_coeff
            
            # パラメータ更新
            self.Wxh -= lr * dWxh
            self.Whh -= lr * dWhh
            self.bh -= lr * dbh
            self.Why -= lr * dWhy
            self.by -= lr * dby
            
            # 次のチャンクに隠れ状態を引き継ぐ
            h = h_states[-1].copy()
        
        return total_loss


# フルBPTT vs Truncated BPTT の比較
np.random.seed(42)

seq_len = 50
n_samples = 16
hidden_dim = 16
n_epochs = 100
lr = 0.01

# データ生成
X_train, Y_train = generate_sequence_data(n_samples, seq_len)

# 各K値で学習
K_values = [5, 10, 25, 50]  # K=50 は実質フルBPTT
results = {}

for K in K_values:
    np.random.seed(42)
    rnn_trunc = RNNWithTruncatedBPTT(input_dim=1, hidden_dim=hidden_dim, output_dim=1)
    losses = []
    
    for epoch in range(n_epochs):
        loss = rnn_trunc.train_step(X_train, Y_train, K=K, lr=lr)
        losses.append(loss)
    
    results[K] = losses

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_k = ['red', 'orange', 'green', 'blue']
for K, color in zip(K_values, colors_k):
    label = f'K={K}' if K < seq_len else f'K={K}（フルBPTT）'
    axes[0].plot(results[K], color=color, linewidth=2, label=label)

axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('損失', fontsize=12)
axes[0].set_title('Truncated BPTT: 異なるK値での学習曲線', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 最終損失の比較
final_losses = [results[K][-1] for K in K_values]
bars = axes[1].bar([f'K={K}' for K in K_values], final_losses, color=colors_k, alpha=0.7)
axes[1].set_xlabel('Truncation長', fontsize=12)
axes[1].set_ylabel('最終損失', fontsize=12)
axes[1].set_title('最終損失の比較', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, final_losses):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("【Truncated BPTT の結果】")
for K in K_values:
    label = '（フルBPTT）' if K >= seq_len else ''
    print(f"  K = {K:3d}{label}: 最終損失 = {results[K][-1]:.6f}")
print()
print("Kが大きいほど勾配がより遠くまで伝播できるが、")
print("小さいKでもsin波のような短期パターンなら十分学習できる。")

---
## 6. 勾配クリッピング

### 6.1 なぜ勾配クリッピングが必要か

勾配爆発が起きると、パラメータの更新量が大きすぎて学習が発散します。

**勾配クリッピング** は、勾配のノルムが閾値 $\theta$ を超えたら、ノルムが $\theta$ になるようにスケーリングする手法です：

$$
\hat{g} = \begin{cases}
g & \text{if } \|g\| \leq \theta \\
\frac{\theta}{\|g\|} g & \text{if } \|g\| > \theta
\end{cases}
$$

### 6.2 クリッピングの種類

| 方式 | 式 | 特徴 |
|-----|---|------|
| ノルムベース | $g \cdot \min(1, \theta / \|g\|)$ | 方向を保持（推奨） |
| 値ベース | $\text{clip}(g, -\theta, \theta)$ | 方向が変わりうる |

In [ ]:
# ============================================================
# 勾配クリッピングの効果を可視化
# ============================================================

np.random.seed(42)

# 勾配爆発が起きやすい設定で比較
seq_len = 50
n_samples = 16
hidden_dim = 32
n_epochs = 150
lr = 0.005

X_train, Y_train = generate_sequence_data(n_samples, seq_len)

# クリッピングなし
np.random.seed(123)
rnn_no_clip = RNNWithBPTT(input_dim=1, hidden_dim=hidden_dim, output_dim=1)
# スペクトル半径を大きめに設定
eigenvalues = np.linalg.eigvals(rnn_no_clip.Whh)
current_sr = np.max(np.abs(eigenvalues))
rnn_no_clip.Whh *= (1.5 / current_sr)

losses_no_clip = []
grad_norms_no_clip = []

for epoch in range(n_epochs):
    outputs = rnn_no_clip.forward(X_train)
    loss = np.mean((outputs - Y_train) ** 2)
    dL = 2.0 * (outputs - Y_train) / (seq_len * n_samples)
    rnn_no_clip.backward(dL)
    
    grads = [rnn_no_clip.dWxh, rnn_no_clip.dWhh, rnn_no_clip.dbh,
             rnn_no_clip.dWhy, rnn_no_clip.dby]
    total_norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
    grad_norms_no_clip.append(total_norm)
    
    # クリッピングなしで更新
    rnn_no_clip.update(lr)
    
    if np.isnan(loss) or loss > 1e6:
        losses_no_clip.append(float('nan'))
        # 残りをNaNで埋める
        losses_no_clip.extend([float('nan')] * (n_epochs - epoch - 1))
        grad_norms_no_clip.extend([float('nan')] * (n_epochs - epoch - 1))
        break
    losses_no_clip.append(loss)

# クリッピングあり
np.random.seed(123)
rnn_clip = RNNWithBPTT(input_dim=1, hidden_dim=hidden_dim, output_dim=1)
eigenvalues = np.linalg.eigvals(rnn_clip.Whh)
current_sr = np.max(np.abs(eigenvalues))
rnn_clip.Whh *= (1.5 / current_sr)

losses_clip = []
grad_norms_clip = []
grad_norms_before_clip = []

for epoch in range(n_epochs):
    outputs = rnn_clip.forward(X_train)
    loss = np.mean((outputs - Y_train) ** 2)
    dL = 2.0 * (outputs - Y_train) / (seq_len * n_samples)
    rnn_clip.backward(dL)
    
    grads = [rnn_clip.dWxh, rnn_clip.dWhh, rnn_clip.dbh,
             rnn_clip.dWhy, rnn_clip.dby]
    total_norm_before = np.sqrt(sum(np.sum(g ** 2) for g in grads))
    grad_norms_before_clip.append(total_norm_before)
    
    # クリッピング
    total_norm_after = rnn_clip.clip_gradients(max_norm=5.0)
    
    grads_after = [rnn_clip.dWxh, rnn_clip.dWhh, rnn_clip.dbh,
                   rnn_clip.dWhy, rnn_clip.dby]
    total_norm_clipped = np.sqrt(sum(np.sum(g ** 2) for g in grads_after))
    grad_norms_clip.append(total_norm_clipped)
    
    rnn_clip.update(lr)
    losses_clip.append(loss)

# 可視化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 学習曲線
axes[0, 0].plot(losses_no_clip, 'r-', linewidth=2, label='クリッピングなし')
axes[0, 0].plot(losses_clip, 'b-', linewidth=2, label='クリッピングあり (max=5.0)')
axes[0, 0].set_xlabel('エポック', fontsize=11)
axes[0, 0].set_ylabel('損失', fontsize=11)
axes[0, 0].set_title('学習曲線', fontsize=13)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim(0, max(5, min(10, max(l for l in losses_no_clip[:50] if not np.isnan(l)))))

# 勾配ノルム
axes[0, 1].semilogy(grad_norms_no_clip, 'r-', linewidth=1.5, alpha=0.7, label='クリッピングなし')
axes[0, 1].semilogy(grad_norms_before_clip, 'b--', linewidth=1.5, alpha=0.5, label='クリッピング前')
axes[0, 1].semilogy(grad_norms_clip, 'b-', linewidth=2, alpha=0.9, label='クリッピング後')
axes[0, 1].axhline(y=5.0, color='gray', linestyle=':', alpha=0.7, label='閾値 = 5.0')
axes[0, 1].set_xlabel('エポック', fontsize=11)
axes[0, 1].set_ylabel('勾配ノルム', fontsize=11)
axes[0, 1].set_title('勾配ノルムの推移', fontsize=13)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# クリッピング前後の比較（ヒストグラム）
valid_before = [x for x in grad_norms_before_clip if not np.isnan(x) and x < 1e10]
valid_after = [x for x in grad_norms_clip if not np.isnan(x)]

axes[1, 0].hist(valid_before, bins=30, alpha=0.6, color='red', label='クリッピング前')
axes[1, 0].hist(valid_after, bins=30, alpha=0.6, color='blue', label='クリッピング後')
axes[1, 0].axvline(x=5.0, color='gray', linestyle=':', linewidth=2, label='閾値 = 5.0')
axes[1, 0].set_xlabel('勾配ノルム', fontsize=11)
axes[1, 0].set_ylabel('頻度', fontsize=11)
axes[1, 0].set_title('勾配ノルムの分布', fontsize=13)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# クリッピングの概念図
ax = axes[1, 1]
theta_circle = np.linspace(0, 2 * np.pi, 100)
max_norm = 5.0

# いくつかの勾配ベクトル例
np.random.seed(0)
for _ in range(8):
    g = np.random.randn(2) * np.random.uniform(1, 10)
    norm = np.linalg.norm(g)
    
    # 元の勾配（赤）
    ax.arrow(0, 0, g[0], g[1], head_width=0.2, head_length=0.15,
             fc='red', ec='red', alpha=0.4, linewidth=1)
    
    # クリッピング後（青）
    if norm > max_norm:
        g_clipped = g * max_norm / norm
    else:
        g_clipped = g
    ax.arrow(0, 0, g_clipped[0], g_clipped[1], head_width=0.2, head_length=0.15,
             fc='blue', ec='blue', alpha=0.7, linewidth=1.5)

# 閾値の円
ax.plot(max_norm * np.cos(theta_circle), max_norm * np.sin(theta_circle),
        'k--', linewidth=2, label=f'閾値 = {max_norm}')

ax.set_xlim(-12, 12)
ax.set_ylim(-12, 12)
ax.set_aspect('equal')
ax.set_xlabel('勾配成分1', fontsize=11)
ax.set_ylabel('勾配成分2', fontsize=11)
ax.set_title('勾配クリッピングの概念図\n(赤:元, 青:クリッピング後)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【勾配クリッピングの効果】")
print("・クリッピングにより勾配の大きさが制限され、学習が安定する")
print("・ノルムベースのクリッピングは勾配の方向を保持する")
print("・注意: 勾配消失問題は解決しない（爆発のみ対処）")

---
## 7. Adding Problem（Hochreiter 1997）

### 7.1 タスクの説明

**Adding Problem** は、RNNの長期依存学習能力を評価するための古典的なベンチマークです（Hochreiter & Schmidhuber, 1997）。

#### 入力
- チャネル0: $[0, 1]$ の一様乱数の系列
- チャネル1: マーカー系列（ほぼ0で、2箇所だけ1）

#### 出力
- マーカーが1の位置にある値の **和**

#### ベースライン
- 常に1.0と予測 → MSE $\approx$ 0.1767
- RNNがこのベースラインを下回れなければ、長期依存を学習できていない

In [ ]:
# ============================================================
# Adding Problem のデータ生成
# ============================================================

def generate_adding_problem(n_samples, seq_length):
    """Adding Problem のデータ生成
    
    Returns:
        X: (seq_length, n_samples, 2) - チャネル0:値, チャネル1:マーカー
        targets: (n_samples, 1) - マークされた値の和
    """
    X = np.zeros((n_samples, seq_length, 2))
    X[:, :, 0] = np.random.uniform(0, 1, (n_samples, seq_length))
    
    # 2つのマーカーを配置（前半と後半に1つずつ）
    for i in range(n_samples):
        idx1 = np.random.randint(0, seq_length // 2)
        idx2 = np.random.randint(seq_length // 2, seq_length)
        X[i, idx1, 1] = 1.0
        X[i, idx2, 1] = 1.0
    
    # ターゲット: マーカー位置の値の和
    targets = np.sum(X[:, :, 0] * X[:, :, 1], axis=1, keepdims=True)
    
    # (seq_length, n_samples, 2) に変換
    X = X.transpose(1, 0, 2)
    
    return X, targets


# タスクの可視化
np.random.seed(42)
seq_len_demo = 50
X_demo, targets_demo = generate_adding_problem(1, seq_len_demo)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

# チャネル0: 値
axes[0].bar(range(seq_len_demo), X_demo[:, 0, 0], color='steelblue', alpha=0.7)
axes[0].set_ylabel('値', fontsize=11)
axes[0].set_title('Adding Problem の入力例', fontsize=13)
axes[0].grid(True, alpha=0.3)

# チャネル1: マーカー
axes[1].bar(range(seq_len_demo), X_demo[:, 0, 1], color='red', alpha=0.7)
axes[1].set_ylabel('マーカー', fontsize=11)
axes[1].set_ylim(-0.1, 1.5)
axes[1].grid(True, alpha=0.3)

# マーカー位置の値をハイライト
marker_indices = np.where(X_demo[:, 0, 1] > 0.5)[0]
marked_values = X_demo[marker_indices, 0, 0]

axes[2].bar(range(seq_len_demo), X_demo[:, 0, 0] * X_demo[:, 0, 1],
            color='green', alpha=0.7)
axes[2].set_ylabel('マークされた値', fontsize=11)
axes[2].set_xlabel('時刻 t', fontsize=11)
axes[2].grid(True, alpha=0.3)

target_val = targets_demo[0, 0]
axes[2].set_title(f'ターゲット = {marked_values[0]:.4f} + {marked_values[1]:.4f} = {target_val:.4f}',
                  fontsize=12)

plt.tight_layout()
plt.show()

# ベースラインの計算
n_test = 10000
_, test_targets = generate_adding_problem(n_test, 50)
baseline_pred = np.ones_like(test_targets) * 1.0
baseline_mse = np.mean((baseline_pred - test_targets) ** 2)
print(f"\n【ベースライン】常に1.0と予測 → MSE = {baseline_mse:.4f}")
print("理論値: E[(X1+X2-1)^2] ≈ 0.1767")
print("RNNがこの値を下回れなければ、長期依存を学習できていない。")

In [ ]:
# ============================================================
# Adding Problem で RNN を訓練
# ============================================================

def train_rnn_adding_problem(seq_length, hidden_dim=32, n_epochs=200,
                             lr=0.01, n_train=256, max_grad_norm=5.0):
    """Adding Problem で RNN を訓練する"""
    rnn = RNNWithBPTT(input_dim=2, hidden_dim=hidden_dim, output_dim=1)
    losses = []
    
    for epoch in range(n_epochs):
        # ミニバッチ生成
        X_batch, targets_batch = generate_adding_problem(n_train, seq_length)
        # targets_batch: (n_train, 1) -> 最後の時刻の出力のみ使用
        
        # 順伝播
        outputs = rnn.forward(X_batch)  # (T, batch, 1)
        
        # 最後の時刻の出力のみを使用
        y_pred = outputs[-1]  # (batch, 1)
        
        # MSE損失
        loss = np.mean((y_pred - targets_batch) ** 2)
        losses.append(loss)
        
        # 逆伝播用の勾配（最後の時刻のみ非ゼロ）
        dL_doutput = np.zeros_like(outputs)
        dL_doutput[-1] = 2.0 * (y_pred - targets_batch) / n_train
        
        rnn.backward(dL_doutput)
        rnn.clip_gradients(max_norm=max_grad_norm)
        rnn.update(lr)
        
        if np.isnan(loss):
            print(f"  NaN at epoch {epoch}, seq_length={seq_length}")
            break
    
    return losses


# 異なる系列長でのトレーニング
np.random.seed(42)
seq_lengths_test = [10, 50, 100, 200]
n_epochs = 300
all_results = {}

print("Adding Problem: RNNの訓練")
print("=" * 50)

for sl in seq_lengths_test:
    np.random.seed(42)
    print(f"系列長 T = {sl} を訓練中...")
    losses = train_rnn_adding_problem(
        seq_length=sl, hidden_dim=32, n_epochs=n_epochs,
        lr=0.005, n_train=128, max_grad_norm=5.0
    )
    all_results[sl] = losses
    final_loss = losses[-1] if not np.isnan(losses[-1]) else float('inf')
    print(f"  最終損失: {final_loss:.6f}")

print("\n訓練完了")

In [ ]:
# ============================================================
# Adding Problem の結果を可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左: 学習曲線
colors_sl = ['green', 'blue', 'orange', 'red']
baseline_mse = 0.1767  # 理論的ベースライン

for sl, color in zip(seq_lengths_test, colors_sl):
    losses = all_results[sl]
    # 移動平均でスムージング
    window = 10
    if len(losses) >= window:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    else:
        smoothed = losses
    axes[0].plot(smoothed, color=color, linewidth=2, label=f'T = {sl}')

axes[0].axhline(y=baseline_mse, color='gray', linestyle='--', linewidth=2,
                label=f'ベースライン (MSE={baseline_mse:.4f})')
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('MSE損失', fontsize=12)
axes[0].set_title('Adding Problem: 学習曲線', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 0.3)

# 右: 最終MSEの比較（棒グラフ）
final_mses = []
for sl in seq_lengths_test:
    losses = all_results[sl]
    # 最後の10エポックの平均
    valid_losses = [l for l in losses[-10:] if not np.isnan(l)]
    if valid_losses:
        final_mses.append(np.mean(valid_losses))
    else:
        final_mses.append(float('nan'))

bars = axes[1].bar([f'T={sl}' for sl in seq_lengths_test], final_mses,
                   color=colors_sl, alpha=0.7, edgecolor='black')
axes[1].axhline(y=baseline_mse, color='gray', linestyle='--', linewidth=2,
                label=f'ベースライン = {baseline_mse:.4f}')

for bar, val in zip(bars, final_mses):
    if not np.isnan(val):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                     f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

axes[1].set_xlabel('系列長', fontsize=12)
axes[1].set_ylabel('最終MSE', fontsize=12)
axes[1].set_title('Adding Problem: 最終MSEの比較', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("【Adding Problem の結果まとめ】")
print("=" * 50)
print(f"ベースライン（常に1.0と予測）: MSE = {baseline_mse:.4f}")
print()
for sl, mse in zip(seq_lengths_test, final_mses):
    if not np.isnan(mse):
        status = '学習成功' if mse < baseline_mse * 0.9 else 'ベースライン未達'
        print(f"  T = {sl:3d}: MSE = {mse:.4f}  [{status}]")
    else:
        print(f"  T = {sl:3d}: 学習失敗（発散）")
print()
print("系列が長くなるほどRNNは長期依存を学習できない。")
print("これがLSTMやGRUが必要になる本質的な理由です。")

---
## 8. まとめと次のステップ

### 8.1 このノートブックで学んだこと

| トピック | ポイント |
|---------|--------|
| BPTTの数式 | $\frac{\partial L}{\partial W_{hh}} = \sum_t \sum_{k=1}^{t} \frac{\partial L_t}{\partial h_t} \prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} \frac{\partial h_k}{\partial W_{hh}}$ |
| 勾配消失 | $\rho(W_{hh}) < 1$ → ヤコビアン積が指数的に減衰 |
| 勾配爆発 | $\rho(W_{hh}) > 1$ → ヤコビアン積が指数的に増大 |
| Truncated BPTT | $K$ ステップ分だけ逆伝播しメモリを節約 |
| 勾配クリッピング | $\hat{g} = g \cdot \min(1, \theta / \|g\|)$ で爆発を防止 |
| Adding Problem | RNNの長期依存学習の限界を実証 |

### 8.2 チートシート

```
BPTT逆伝播の核心:
  dh_next ← 出力層からの勾配 + 未来の時刻からの勾配
  dtanh   ← dh_next * (1 - h_t^2)
  dWhh    += h_{t-1}^T @ dtanh
  dh_prev ← dtanh @ Whh^T

勾配クリッピング:
  norm = sqrt(sum(g_i^2))
  if norm > theta: g *= theta / norm

Truncated BPTT:
  for chunk in chunks(sequence, K):
    forward(chunk)   # 隠れ状態は引き継ぐ
    backward(chunk)  # 勾配はチャンク内のみ
    update()         # パラメータ更新
```

### 8.3 よくある間違い

| 間違い | 正しい理解 |
|--------|----------|
| 勾配消失 = 勾配が0 | 勾配が **指数的に小さく** なること。完全に0ではない |
| クリッピングで消失も解決 | クリッピングは **爆発** のみ対処。消失にはLSTM/GRUが必要 |
| Truncated BPTTで長期依存OK | K以上離れた依存は学習 **不可能** |
| スペクトル半径=1なら安定 | tanh の導関数 ($\leq 1$) もあるため、実際は消失しやすい |

### 8.4 学習目標チェックリスト

- [ ] BPTTの数式を書き下し、連鎖律がどう適用されるか説明できる
- [ ] NumPyでRNNのforward/backwardを実装し、数値勾配チェックに通せる
- [ ] $W_{hh}$ のスペクトル半径と勾配消失/爆発の関係を説明できる
- [ ] Truncated BPTTの実装と、フルBPTTとのトレードオフを説明できる
- [ ] 勾配クリッピングを実装できる
- [ ] Adding Problemで、RNNが長い系列で失敗することを実証できる

### 8.5 次のステップ

**Notebook 123: LSTM ― ゲート機構による勾配消失の解決**

このノートブックで明らかになった勾配消失問題に対する解決策が **LSTM（Long Short-Term Memory）** です。
LSTMは「セル状態」と「ゲート機構」により、勾配のハイウェイを構築します。

---
## 自己評価クイズ

### Q1. BPTTで $W_{hh}$ の勾配を計算する際、なぜ単純な逆伝播ではなく時間を遡る必要があるのですか？

<details>
<summary>回答を表示</summary>

RNNでは同じ重み $W_{hh}$ が全時刻で共有されています。時刻 $t$ の隠れ状態 $h_t$ は、$W_{hh}$ を介して $h_{t-1}$, $h_{t-2}$, ... に間接的に依存しています。したがって、$\partial L / \partial W_{hh}$ を正確に計算するには、$W_{hh}$ が各時刻で使われた全ての箇所からの勾配寄与を **蓄積** する必要があります。これが二重和 $\sum_t \sum_{k=1}^{t}$ の由来です。

</details>

### Q2. $W_{hh}$ のスペクトル半径が0.8のとき、50時刻前の情報からの勾配はおよそどれくらい減衰しますか？

<details>
<summary>回答を表示</summary>

スペクトル半径 $\rho = 0.8$ のとき、50時刻を遡ると勾配はおよそ $0.8^{50} \approx 1.4 \times 10^{-5}$ 倍に減衰します。さらに $\tanh$ の導関数 ($\leq 1$) も掛かるため、実際はさらに小さくなります。これにより、50時刻前の情報はほぼ学習に寄与できません。

</details>

### Q3. Truncated BPTTで K=10 とした場合、系列長T=100のデータに対して何が保証されますか？何が失われますか？

<details>
<summary>回答を表示</summary>

**保証されること**: 10時刻以内の短期依存関係は正確に学習できます。メモリ使用量が $O(K \cdot H)$ に抑えられます。

**失われること**: 10時刻より遠い長期依存関係の勾配は計算されません。つまり、「20時刻前の入力がどのように今の出力に影響するか」は学習できません。順伝播では隠れ状態が全時刻を通じて伝播するため、長期の情報は **暗黙的に** 保持されますが、逆伝播による **明示的な** 学習はできません。

</details>

### Q4. 勾配クリッピングは勾配消失問題を解決しますか？理由を述べてください。

<details>
<summary>回答を表示</summary>

**解決しません。** 勾配クリッピングは勾配のノルムが大きすぎるとき（爆発時）にスケールダウンするものです。勾配が小さすぎる場合（消失時）には何も行いません。

勾配消失の解決には、アーキテクチャレベルでの変更が必要です：
- **LSTM**: セル状態への加法的な更新により勾配のハイウェイを構築
- **GRU**: リセットゲートと更新ゲートで勾配の流れを制御
- **残差接続**: $h_t = f(h_{t-1}) + h_{t-1}$ で恒等写像を学習しやすくする

</details>

### Q5. Adding Problemで系列長を200にしたとき、バニラRNNのMSEがベースライン(約0.1767)を下回れない理由を説明してください。

<details>
<summary>回答を表示</summary>

Adding Problemでは、系列の前半のどこかにあるマーカー位置の値を、最後の出力まで **記憶** し続ける必要があります。T=200の場合、最大100時刻以上前の情報を保持する必要があります。

BPTTの逆伝播では、この100時刻以上のヤコビアン積が指数的に減衰するため、「どの値を記憶すべきか」を学習するための勾配がほぼゼロになります。結果として、RNNは「とりあえず平均値(1.0)を出力する」という戦略から脱出できず、ベースラインと同等のMSEにとどまります。

</details>

---
## 参考文献

1. Werbos, P. J. (1990). *Backpropagation Through Time: What It Does and How to Do It.* Proceedings of the IEEE, 78(10), 1550-1560.
2. Hochreiter, S. (1991). *Untersuchungen zu dynamischen neuronalen Netzen.* Diploma thesis, TU Munich.
3. Bengio, Y., Simard, P., & Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent is Difficult.* IEEE Transactions on Neural Networks, 5(2), 157-166.
4. Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory.* Neural Computation, 9(8), 1735-1780.
5. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). *On the Difficulty of Training Recurrent Neural Networks.* ICML 2013.
6. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.* MIT Press. Chapter 10: Sequence Modeling.
7. 斎藤康毅 (2018). 『ゼロから作るDeep Learning 2 ― 自然言語処理編』. O'Reilly Japan.